# 09. Ruído local revisado

O notebook 08 não avaliou de fato a técnica de ruído local: com o MAD do resíduo como medida de dispersão, o mapa ficou indefinido em todas as imagens do train, porque em boa parte do papel o resíduo (imagem menos mediana 3 x 3) vale exatamente zero e o MAD de um bloco plano é zero. Esta versão muda duas coisas, fixadas em `results/grade_ruido_local.json` antes de qualquer execução:

1. a dispersão do bloco é o **desvio-padrão** do resíduo nos pixels de fundo, com o valor `log(sigma + 0,1)` (o piso de 0,1 evita o log de zero em papel plano);
2. o z de cada bloco é **calibrado** pela média e pelo desvio desse valor nos blocos válidos das imagens **autênticas do train**, separadamente para cada tamanho de bloco B (32 e 64).

O fundo é o que fica fora da máscara de Otsu do texto, dilatada de 3 px (elemento 7 x 7; Otsu e dilatação são métodos de apoio e não aparecem nos slides da disciplina). Blocos com menos de 50% de fundo são ignorados. Como o z é calibrado por um conjunto fixo, e não pela própria imagem, ele é de duas pontas: um bloco muito mais plano ou muito mais ruidoso que o papel típico fica com |z| alto. A feature de imagem é a fração de blocos válidos com |z| > 3.

Etapa 1: mede-se se o mapa destaca a região adulterada acima do acaso, no split `train`. Etapa 2, só se algum B passar: classificador candidato. O split `test` do *Find it again!* não é aberto. Seed 42. Sem paralelismo. Os números aparecem apenas como saída das células; este notebook não registra conclusões.

In [1]:
import sys
sys.path.append("../src")
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from classification import bootstrap_auc, cv_evaluate
from comparacao_cv import auc_por_particao, diferenca_pareada
from dataset_loader import load_dataset, load_gray
from features import FEATURE_NAMES, load_or_extract
from features_v2 import V2_NAMES, load_or_extract_v2
from ruido_fundo import TAMANHOS, auc_regioes, hit_rate_evaluation
from ruido_local import calibrar, fracao_suspeita, mapa_z, valores_ruido_local

RAIZ = Path("..")
DATASET_ROOT = str(RAIZ / "data/findit2")

## 1. Grade e critérios congelados

A célula confere que o código novo (`src/ruido_local.py`), o `src/ruido_fundo.py` (reutilizado sem alteração, para a AUC de região) e o `src/comparacao_cv.py` ainda têm os hashes registrados.

In [2]:
grade = json.loads((RAIZ / "results/grade_ruido_local.json").read_text(encoding="utf-8"))
print("congelado em:", grade["congelado_em"])
esperados = {**grade["codigo"], "src/ruido_fundo.py": grade["reutiliza_sem_alterar"]["src/ruido_fundo.py"],
             "src/comparacao_cv.py": grade["reutiliza_sem_alterar"]["src/comparacao_cv.py"]}
for arquivo, esperado in esperados.items():
    print(arquivo, "idêntico ao congelado:", hashlib.sha256((RAIZ / arquivo).read_bytes()).hexdigest() == esperado)
print(json.dumps(grade["tecnica"], ensure_ascii=False, indent=1))

congelado em: 2026-11-06T17:04:46
src/ruido_local.py idêntico ao congelado: True
src/ruido_fundo.py idêntico ao congelado: True
src/comparacao_cv.py idêntico ao congelado: True
{
 "residuo": "imagem - mediana 3x3 (copy_move_residuo.residuo), tons de cinza, resolução original",
 "fundo": "fora da máscara de Otsu do texto dilatada de 3 px (interpretação: elemento 7x7, raio 3 px; a rodada anterior usou 5x5)",
 "blocos": "lado B em {32, 64}, só blocos inteiros; ignora bloco com menos de 50% de fundo",
 "sigma": "desvio-padrão (ddof = 0) do resíduo nos pixels de fundo do bloco",
 "valor": "log(sigma + 0,1)",
 "calibracao": "média e desvio do valor nos blocos válidos (agrupados) das imagens autênticas do train, separados por B",
 "mapa": "z = (valor - média) / desvio, por bloco válido",
 "feature_de_imagem": "fração de blocos válidos com |z| > 3 (uma por B); sem bloco válido: indefinida, imputada pela mediana e contada",
 "paralelismo": "nenhum (sequencial); sem fork"
}


## 2. Etapa 1: calibração e mapas no train

Calculam-se, em série, os valores `log(sigma + 0,1)` por bloco de todas as imagens do split `train`, para B = 32 e B = 64. A média e o desvio de calibração, por B, vêm só dos blocos válidos das imagens autênticas. Em seguida, cada imagem recebe o seu mapa z. Reportam-se os parâmetros de calibração e a distribuição de |z| nas autênticas e nas adulteradas (as adulteradas não participam da calibração).

In [3]:
train = [s for s in load_dataset(DATASET_ROOT, "train") if s.image_path.exists()]
valores = {str(s.image_path): {b: valores_ruido_local(load_gray(s.image_path), b) for b in TAMANHOS} for s in train}
autenticas = [str(s.image_path) for s in train if not s.is_tampered]
calibracao = {b: calibrar([valores[caminho][b] for caminho in autenticas]) for b in TAMANHOS}
for b, (media, desvio) in calibracao.items():
    n_blocos = sum(int((~np.isnan(valores[c][b])).sum()) for c in autenticas if valores[c][b] is not None)
    print(f"B={b}: média = {media:.3f} | desvio = {desvio:.3f} | blocos válidos nas autênticas = {n_blocos}")

aviso: 1 imagens com anotação ausente/malformada tratadas como regions=[] (train)
aviso: 2 imagens com forged divergente da anotação (train): ['X51008114260.png', 'X51006619709.png']
aviso: 2 regiões marcadas como área original mas com modificação real (train)


B=32: média = -1.914 | desvio = 0.961 | blocos válidos nas autênticas = 2771287
B=64: média = -1.760 | desvio = 1.086 | blocos válidos nas autênticas = 693024


In [4]:
mapas = {caminho: {f"local_{b}": mapa_z(v[b], *calibracao[b]) for b in TAMANHOS} for caminho, v in valores.items()}
print("mapas indefinidos (sem bloco válido):", {f"local_{b}": sum(m[f"local_{b}"] is None for m in mapas.values()) for b in TAMANHOS})
linhas = []
for b in TAMANHOS:
    for classe, selecao in (("autênticas", [str(s.image_path) for s in train if not s.is_tampered]),
                            ("adulteradas", [str(s.image_path) for s in train if s.is_tampered])):
        fracoes = [fracao_suspeita(mapas[c][f"local_{b}"]) for c in selecao]
        fracoes = [f for f in fracoes if f is not None]
        linhas.append({"B": b, "classe": classe, "n": len(fracoes), "fracao_suspeita_media": np.mean(fracoes),
                       "mediana": np.median(fracoes), "p90": np.percentile(fracoes, 90)})
print(pd.DataFrame(linhas).round(4).to_string(index=False))

mapas indefinidos (sem bloco válido): {'local_32': 0, 'local_64': 0}
 B      classe   n  fracao_suspeita_media  mediana    p90
32  autênticas 483                 0.0707   0.0227 0.1803
32 adulteradas  94                 0.0745   0.0185 0.1834
64  autênticas 483                 0.0084   0.0000 0.0042
64 adulteradas  94                 0.0123   0.0000 0.0039


## 3. Etapa 1: AUC no nível de região

O critério mede se o mapa destaca a região adulterada acima de pseudo-regiões sorteadas ao acaso. Em cada imagem adulterada, a região real vale o maior |z| entre os blocos que a tocam; as pseudo-regiões têm o tamanho da maior região modificada, contêm texto e não se sobrepõem às regiões anotadas (as regras de `evaluate_region_technique`, 20 sorteios por imagem, seed 42). A AUC da imagem é a fração de pseudo-regiões com score menor que o da região real, e a AUC final é a média entre imagens, com IC de bootstrap (2000 reamostragens sobre as imagens). Como complemento, reporta-se o acerto real contra o acaso calculado por `evaluate_region_technique`. A técnica passa se o limite inferior do IC for maior que 0,5 para algum B.

In [5]:
linhas = []
for b in TAMANHOS:
    r = auc_regioes(train, mapas, "local", b)
    h = hit_rate_evaluation([s for s in train if s.is_tampered and s.regions], mapas, "local", b)
    linhas.append({**{k: r[k] for k in ("tecnica", "B", "n_imagens", "auc", "ic_min", "ic_max")}, "exclusoes": r["exclusoes"],
                   "acerto_real": h["real_hit_rate"], "acerto_acaso": h["chance_hit_rate"], "n_evaluate": h["n_samples"]})
tabela = pd.DataFrame(linhas)
pd.set_option("display.width", 250)
print(tabela.drop(columns=["exclusoes"]).round(3).to_string(index=False))
print(tabela[["B", "exclusoes"]].to_string(index=False))
passam = tabela[tabela["ic_min"] > 0.5].sort_values(["auc", "B"], ascending=[False, True])
vencedor = int(passam.iloc[0]["B"]) if len(passam) else None
print("B vencedor:", vencedor, "| nenhum B passou (parar):", vencedor is None)

tecnica  B  n_imagens   auc  ic_min  ic_max  acerto_real  acerto_acaso  n_evaluate
  local 32         82 0.647   0.581   0.711        0.667         0.474          72
  local 64         83 0.641   0.579   0.701        0.606         0.491          71
 B                                                                      exclusoes
32 {'mapa_indefinido': 0, 'regiao_real_sem_bloco': 7, 'poucas_pseudo_validas': 4}
64 {'mapa_indefinido': 0, 'regiao_real_sem_bloco': 7, 'poucas_pseudo_validas': 3}
B vencedor: 32 | nenhum B passou (parar): False


Registro da etapa de região em `results/diagnostico_ruido_local_train.json`.

In [6]:
saida = {"escopo": "só train do Find it again!", "calibracao": {str(b): {"media": m, "desvio": s} for b, (m, s) in calibracao.items()},
         "regiao": tabela.to_dict("records"), "B_vencedor": vencedor, "parar": vencedor is None, "codigo_sha256": grade["codigo"]}
(RAIZ / "results/diagnostico_ruido_local_train.json").write_text(json.dumps(saida, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("gravado")

gravado


## 4. Etapa 2: classificador candidato

Pelo critério da etapa de região, os dois B passaram, e o B = 32 tem a maior AUC. O candidato usa as 12 features do classificador v2 mais uma feature: a fração de blocos válidos com |z| > 3, com B = 32. Os dados são os mesmos do v2, train + val do *Find it again!* (o `test` não é aberto). O z das imagens de val usa a calibração do train autêntico, a mesma da etapa de região. A feature é guardada em `data/features_ruido_local_train.csv` e `data/features_ruido_local_val.csv` (fora do Git). Imagens com feature indefinida (se houver) recebem a mediana das definidas, e a contagem é reportada.

Ressalva do critério: a calibração (duas médias e dois desvios) usa as autênticas do train, que na validação cruzada também aparecem nos folds de validação. É pouca informação, mas depende do rótulo, o que pode favorecer o candidato.

In [7]:
amostras = {"train": train, "val": load_dataset(DATASET_ROOT, "val")}
tabelas = {}
for split, S in amostras.items():
    linhas = []
    for s in S:
        v = valores[str(s.image_path)] if split == "train" else {b: valores_ruido_local(load_gray(s.image_path), b) for b in TAMANHOS}
        linhas.append({"image": s.image_path.name, **{f"ruido_local_{b}_frac": fracao_suspeita(mapa_z(v[b], *calibracao[b])) for b in TAMANHOS}})
    novas = pd.DataFrame(linhas)
    novas.to_csv(RAIZ / f"data/features_ruido_local_{split}.csv", index=False)
    v1 = load_or_extract(S, str(RAIZ / f"data/features_{split}.csv"))
    v2 = load_or_extract_v2(S, str(RAIZ / f"data/features_v2_{split}.csv"))
    tabelas[split] = v1.merge(v2, on="image", validate="one_to_one").merge(novas, on="image", validate="one_to_one")
dados = pd.concat([tabelas["train"], tabelas["val"]], ignore_index=True)
NOVA = [f"ruido_local_{vencedor}_frac"]
indefinidas = int(dados[NOVA].isna().any(axis=1).sum())
dados[NOVA] = dados[NOVA].fillna(dados[NOVA].median())
y = dados["is_tampered"].to_numpy()
print("train + val:", len(dados), "| adulteradas:", int(y.sum()), "| autênticas:", int((y == 0).sum()))
print("imagens com feature indefinida (imputadas):", indefinidas)
print(dados.groupby("is_tampered")[NOVA].agg(["mean", "median"]).rename(index={0: "autêntica", 1: "adulterada"}).round(4).to_string())

aviso: 1 imagens listadas em val.txt e ausentes de val/ foram ignoradas: ['X51006619709.png']
aviso: 1 regiões marcadas como área original mas com modificação real (val)


train + val: 769 | adulteradas: 127 | autênticas: 642
imagens com feature indefinida (imputadas): 0
            ruido_local_32_frac        
                           mean  median
is_tampered                            
autêntica                0.0733  0.0223
adulterada               0.0735  0.0210


Reprodução do protocolo do v2: validação cruzada estratificada, 5 folds x 3 repetições, seed 42, com padronização e regressão logística balanceada. `auc_por_particao` repete o divisor e a semente de `cv_evaluate` para obter a AUC de cada partição. A célula confere que a média reproduz a AUC externa registrada em `results/configuracao_congelada.json` (0,665416 com desvio 0,046784).

In [8]:
modelo = lambda: make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
colunas_v2 = FEATURE_NAMES + V2_NAMES
colunas_candidato = colunas_v2 + NOVA
config = json.loads((RAIZ / "results/configuracao_congelada.json").read_text(encoding="utf-8"))
auc_v2 = auc_por_particao(modelo, dados[colunas_v2].to_numpy(), y)
resumo_v2 = cv_evaluate(modelo, dados[colunas_v2].to_numpy(), y)
print("v2: média das partições = %.6f | desvio = %.6f | cv_evaluate: %.6f +- %.6f | registrado: %.6f +- %.6f" % (
    auc_v2.mean(), auc_v2.std(), resumo_v2["auc_mean"], resumo_v2["auc_std"],
    config["auc_externa_cv"]["media"], config["auc_externa_cv"]["desvio"]))
reproduz = bool(abs(auc_v2.mean() - config["auc_externa_cv"]["media"]) < 1e-9)
print("reproduz o registrado:", reproduz)

v2: média das partições = 0.665416 | desvio = 0.046784 | cv_evaluate: 0.665416 +- 0.046784 | registrado: 0.665416 +- 0.046784
reproduz o registrado: True


Comparação lado a lado, nas mesmas 15 partições. A diferença pareada é AUC(candidato) menos AUC(v2) em cada partição. O IC 95% usa a correção de Nadeau e Bengio para validação cruzada repetida; o IC ingênuo é mostrado só para comparação.

In [9]:
auc_cand = auc_por_particao(modelo, dados[colunas_candidato].to_numpy(), y)
comparacao = diferenca_pareada(auc_cand, auc_v2)
por_particao = pd.DataFrame({"v2": auc_v2, "candidato": auc_cand, "diferenca": auc_cand - auc_v2})
print(por_particao.round(4).to_string())
print("v2:        AUC = %.4f +- %.4f" % (auc_v2.mean(), auc_v2.std()))
print("candidato: AUC = %.4f +- %.4f" % (auc_cand.mean(), auc_cand.std()))
print("diferença pareada média = %.4f (desvio entre partições %.4f)" % (comparacao["diferenca_media"], comparacao["desvio"]))
print("IC 95%% corrigido (Nadeau e Bengio): [%.4f, %.4f]" % tuple(comparacao["ic95_corrigido"]))
print("IC 95%% ingênuo: [%.4f, %.4f]" % tuple(comparacao["ic95_ingenuo"]))
print("partições em que o candidato supera o v2:", int((auc_cand > auc_v2).sum()), "de", len(auc_v2))

        v2  candidato  diferenca
0   0.6412     0.6406    -0.0006
1   0.6983     0.6998     0.0015
2   0.6195     0.6248     0.0053
3   0.6695     0.6716     0.0022
4   0.6909     0.7141     0.0231
5   0.7374     0.7434     0.0060
6   0.6581     0.6334    -0.0246
7   0.6028     0.6152     0.0124
8   0.7681     0.7566    -0.0115
9   0.5925     0.6175     0.0250
10  0.6623     0.6581    -0.0042
11  0.7058     0.7064     0.0006
12  0.6623     0.6664     0.0040
13  0.6322     0.6620     0.0298
14  0.6403     0.6372    -0.0031
v2:        AUC = 0.6654 +- 0.0468
candidato: AUC = 0.6698 +- 0.0435
diferença pareada média = 0.0044 (desvio entre partições 0.0141)
IC 95% corrigido (Nadeau e Bengio): [-0.0126, 0.0214]
IC 95% ingênuo: [-0.0034, 0.0122]
partições em que o candidato supera o v2: 10 de 15


AUC isolada da feature nova (adulterada contra autêntica, nas 769 imagens de train e val), com IC de bootstrap sobre as imagens (2000 reamostragens, seed 42). Uma AUC de 0,5 é o acaso. Como as autênticas do train entram na calibração, a AUC isolada também carrega a pequena vantagem descrita acima.

In [10]:
isoladas = {}
for nome in NOVA:
    valor = dados[nome].to_numpy()
    isoladas[nome] = {"auc": float(roc_auc_score(y, valor)), "ic_bootstrap": list(bootstrap_auc(y, valor))}
    print(nome, "AUC = %.3f, IC bootstrap [%.3f, %.3f]" % (isoladas[nome]["auc"], *isoladas[nome]["ic_bootstrap"]))
so_train = dados.iloc[: len(tabelas["train"])]
so_val = dados.iloc[len(tabelas["train"]):]
print("AUC isolada só no val (sem autênticas usadas na calibração): %.3f" % roc_auc_score(so_val["is_tampered"], so_val[NOVA[0]]))

ruido_local_32_frac AUC = 0.507, IC bootstrap [0.452, 0.562]
AUC isolada só no val (sem autênticas usadas na calibração): 0.535


Critério congelado: o candidato vence se a AUC externa média na validação cruzada for pelo menos 0,70 **e** o limite inferior do IC 95% corrigido da diferença pareada contra o v2 for maior que zero.

In [11]:
atingiu_auc = bool(auc_cand.mean() >= 0.70)
atingiu_diferenca = bool(comparacao["ic95_corrigido"][0] > 0)
print("AUC externa do candidato >= 0,70:", atingiu_auc, "(%.4f)" % auc_cand.mean())
print("limite inferior do IC95 corrigido da diferença > 0:", atingiu_diferenca, "(%.4f)" % comparacao["ic95_corrigido"][0])
print("critério atingido:", atingiu_auc and atingiu_diferenca)
saida = {"escopo": "train + val do Find it again!; test não aberto", "n_imagens": int(len(dados)), "feature_nova": NOVA, "B_vencedor": vencedor,
         "calibracao": {str(b): {"media": m, "desvio": s_} for b, (m, s_) in calibracao.items()},
         "imagens_com_feature_indefinida": indefinidas,
         "v2": {"auc_media": float(auc_v2.mean()), "auc_desvio": float(auc_v2.std()), "auc_por_particao": [float(v) for v in auc_v2], "reproduz_registrado": reproduz},
         "candidato": {"auc_media": float(auc_cand.mean()), "auc_desvio": float(auc_cand.std()), "auc_por_particao": [float(v) for v in auc_cand]},
         "diferenca_pareada": comparacao, "auc_isolada": isoladas,
         "criterio": {"auc_externa_minima": 0.70, "atingiu_auc": atingiu_auc, "atingiu_diferenca": atingiu_diferenca, "atingido": atingiu_auc and atingiu_diferenca},
         "codigo_sha256": grade["codigo"]}
(RAIZ / "results/classificador_ruido_local_cv.json").write_text(json.dumps(saida, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("gravado")

AUC externa do candidato >= 0,70: False (0.6698)
limite inferior do IC95 corrigido da diferença > 0: False (-0.0126)
critério atingido: False
gravado
